# Stage 01 — Frame: weekly demand forecast per store x article
Goal: confirm target, unit of analysis and baselines using the raw file.

In [1]:
import pandas as pd
d = pd.read_csv("data/raw/perishable_45k.csv", low_memory=False, parse_dates=["BILL_DATE"])
print(d.shape); print(d.dtypes.head(20))

(45000, 41)
SITE                      object
SITE_NAME                 object
BILL_DATE         datetime64[ns]
BILL_TIME                 object
BILL_TYPE                 object
BILL_NO                   object
CUSTOMER_NAME             object
MOBILE                   float64
MOBILE_HASH               object
SALES_CHANNEL             object
PAYMENT_METHOD            object
CREATED_BY                object
DIVISION                  object
SECTION                   object
ARTICLE                   object
BRAND                     object
BARCODE                    int64
HSN_CODE                   int64
UOM                       object
BILL_QUANTITY              int64
dtype: object


In [2]:
sale = d[d.BILL_TYPE.eq("Sale")]
sale = sale.assign(week=sale.BILL_DATE.dt.to_period("W-SUN").dt.start_time)
print("sale rows", len(sale), "qty==0:", (sale.BILL_QUANTITY == 0).sum())
sale = sale[sale.BILL_QUANTITY > 0]
# store code hygiene: SITE (10 codes) vs SITE_NAME (5)
print(d.groupby("SITE_NAME").SITE.unique())

sale rows 33540 qty==0: 2773
SITE_NAME
East Fresh Store     [East-Fresh, TBD, UNKNOWN, nan, LOC99, STORE_X...
Metro Fresh Store    [Metro-Fresh, STORE_X, LOC99, nan, TBD, UNKNOW...
RVM1 Fresh Store     [RVM1-Fresh, UNKNOWN, STORE_X, BRANCH-01, nan,...
RVM2 Fresh Store     [RVM2-Fresh, BRANCH-01, UNKNOWN, TBD, LOC99, n...
West Fresh Store     [West-Fresh, UNKNOWN, nan, TBD, BRANCH-01, LOC...
Name: SITE, dtype: object


In [3]:
w = sale.groupby(["SITE_NAME", "ARTICLE", "week"]).BILL_QUANTITY.sum().rename("weekly_units").reset_index()
print("series:", w.groupby(["SITE_NAME", "ARTICLE"]).ngroups, "weeks:", w.week.nunique())
full = w.groupby(["SITE_NAME", "ARTICLE"]).week.nunique()
print("weeks with sales per series (of ~53): median", full.median(), "min", full.min())
print("train weeks <2025-01-01:", (w.week < "2025-01-01").sum(), "test:", (w.week >= "2025-01-01").sum())

series: 145 weeks: 53
weeks with sales per series (of ~53): median 50.0 min 26
train weeks <2025-01-01: 5363 test: 1704


In [4]:
# naive baselines on the proposed split (missing weeks = 0 units)
import numpy as np
g = w.set_index(["SITE_NAME","ARTICLE","week"]).weekly_units
wide = g.unstack("week").fillna(0)
test = [c for c in wide.columns if c >= pd.Timestamp("2025-01-01")]
last = wide.shift(1, axis=1)[test]
yt = wide[test]
print("last-week WAPE:", round((yt - last).abs().sum().sum() / yt.sum().sum(), 3))

last-week WAPE: 0.674


In [5]:
# seasonal-naive (same week previous year) is impossible with 1 year of data -> use last-week and trailing-4-week mean
m4 = wide.T.rolling(4).mean().shift(1).T[test]
print("trailing-4wk-mean WAPE:", round((yt - m4).abs().sum().sum() / yt.sum().sum(), 3))
print("mean weekly units per series:", round(yt.values.mean(), 1))

trailing-4wk-mean WAPE: 0.548
mean weekly units per series: 10.6
